Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from astropy.io import fits
from astropy.table import Table


import pandas as pd
from scipy.signal import savgol_filter
from scipy import interpolate

import seaborn as sns


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, LSTM
from tensorflow.keras import models, regularizers
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import ReduceLROnPlateau
from keras.layers import Dense, BatchNormalization
from tensorflow.keras.layers import Dropout
from tensorflow.keras.initializers import HeNormal,GlorotNormal
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.model_selection import train_test_split, KFold, StratifiedShuffleSplit
from sklearn.utils import resample

import csv

from IPython.display import clear_output

from tensorflow.keras.models import load_model
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


# Loading data

In this section we load and preproces spectra data of ET galaxies, we tagged with "0" label for this class.

In [ ]:

daty=[]
datx=[]
label=[]
spcid=[]
z_a=[]
sc_a=[]
sac_a=[]
edgeon_a=[]
dk_a=[]
merger_a=[]
for p in contenido[5:1005]:
    
    data= fits.open("/Users/Diego/Documents/SDSS/Espirales/"+p) 
    
    IMG = data[1].data
    hdr=data[0].header
    
    if ((max(10**(IMG.loglam))>9130)and(min(10**(IMG.loglam))<3840)):
    
        f = interpolate.interp1d(10**(IMG.loglam),IMG.flux)
        
        xnew = np.arange(3840, 9140, 10)
        
        #Interquartil__________________________________
        q3,q1=np.percentile(IMG.flux,[75, 25])
        
        m=np.percentile(IMG.flux,[50])
        ynew=(f(xnew)-m)/(q3-q1)
        
        
        daty.append(ynew)
        datx.append(xnew)
        label.append(0)#________________________________Se usa etiqueta 0 para espirales
        spcid.append(hdr['SPEC_ID'])
        if df['specObjID'].isin([int(hdr['SPEC_ID'])]).any()==1:
            z=df[df.specObjID==int(hdr['SPEC_ID'])].z.values[0]
            sc=df[df.specObjID==int(hdr['SPEC_ID'])].spiralclock.values[0]
            sac=df[df.specObjID==int(hdr['SPEC_ID'])].spiralanticlock.values[0]
            edgeon=df[df.specObjID==int(hdr['SPEC_ID'])].edgeon.values[0]
            dk=df[df.specObjID==int(hdr['SPEC_ID'])].dontknow.values[0]
            merger=df[df.specObjID==int(hdr['SPEC_ID'])].merger.values[0]
        z_a.append(z)
        sc_a.append(sc)
        sac_a.append(sac)
        edgeon_a.append(edgeon)
        dk_a.append(dk)
        merger_a.append(merger)
        #print(hdr['SPEC_ID'])
        
    data.close()

dy=np.array(daty)
dx=np.array(datx)
lbl=np.array(label)
z_l=np.array(z_a)
sc_l=np.array(sc_a)
sac_l=np.array(sac_a)
edgeon_l=np.array(edgeon_a)
dk_l=np.array(dk_a)
merger_l=np.array(merger_a)
spcid_l=np.array(spcid)

The same for LT

In [ ]:

dately=[]
datelx=[]
labelel=[]
contador=0
spcidel=[]
zel_a=[]
scel_a=[]
sacel_a=[]
edgeonel_a=[]
dkel_a=[]
mergerel_a=[]
for p in contenido2[5:1005]:
    data= fits.open("/Users/Diego/Documents/SDSS/Elipticas/"+p) 
    IMG = data[1].data
    hdr=data[0].header
    if ((max(10**(IMG.loglam))>9130)and(min(10**(IMG.loglam))<3840)):
        contador=contador + 1
        #y_filtered = savgol_filter(IMG.flux, 15, 5)
       # print(p)
        f = interpolate.interp1d(10**(IMG.loglam),IMG.flux)
        #f = interpolate.interp1d(10**(IMG.loglam),y_filtered)
        xnew = np.arange(3840, 9140, 10)
        #ynew=(f(xnew))/max(IMG.flux)
        #Interquartil
        #q3,q1=np.percentile(IMG.flux,[75, 25])
        #m=np.percentile(IMG.flux,[50])
        #ynew=(f(xnew)-m)/(q3-q1)
        ynew=f(xnew)
        
        
        dately.append(ynew)
        datelx.append(xnew)
        labelel.append(1)  #______________Se usa 1 para elipticas
        spcidel.append(hdr['SPEC_ID'])
        if df2['specObjID'].isin([int(hdr['SPEC_ID'])]).any()==1:
        #if(df2[df2.eq(int(hdr['SPEC_ID'])).any(1)].shape[0]==1):
            #print("Spec ID"+str(hdr['SPEC_ID'])+" z = "+str(float(df.at[df[df.specObjID==int(hdr['SPEC_ID'])].index[0],'z']))+" Archivo "+p)
            #print(df[df.specObjID==int(hdr['SPEC_ID'])].z.values[0].size)
            #spiralanticlock	edgeon	dontknow	merger
            z=df2[df2.specObjID==int(hdr['SPEC_ID'])].z.values[0]
            sc=df2[df2.specObjID==int(hdr['SPEC_ID'])].spiralclock.values[0]
            sac=df2[df2.specObjID==int(hdr['SPEC_ID'])].spiralanticlock.values[0]
            edgeon=df2[df2.specObjID==int(hdr['SPEC_ID'])].edgeon.values[0]
            dk=df2[df2.specObjID==int(hdr['SPEC_ID'])].dontknow.values[0]
            merger=df2[df2.specObjID==int(hdr['SPEC_ID'])].merger.values[0]
            #print(z)
            #print(sc)
        zel_a.append(z)
        scel_a.append(sc)
        sacel_a.append(sac)
        edgeonel_a.append(edgeon)
        dkel_a.append(dk)
        mergerel_a.append(merger)
    data.close()
dely=np.array(dately)
delx=np.array(datelx)
lblel=np.array(labelel)
zel_l=np.array(zel_a)
scel_l=np.array(scel_a)
sacel_l=np.array(sacel_a)
edgeonel_l=np.array(edgeonel_a)
dkel_l=np.array(dkel_a)
mergerel_l=np.array(mergerel_a)
spcidel_l=np.array(spcidel)
print('Numero de espectros desechados = ' + str(-contador+1000))

# Stacking elements

Overall hyperparameters

In [ ]:
drop=0.2
paciencia=50
reduce_lr = ReduceLROnPlateau(monitor='redshift_out_loss', factor=0.2, patience=3, min_lr=0.000001)
early_stopping = EarlyStopping(monitor='val_accuracy', patience=paciencia, restore_best_weights=True)
reduce_lr_1 = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=5, min_lr=0.000001)
early_stopping_1 = EarlyStopping(monitor='val_class_out_accuracy', patience=paciencia, restore_best_weights=True)
k_size=10


## Convolutives NN

In [ ]:
def net_cnn_1():
    network_cnn_1 = models.Sequential()
    network_cnn_1.add(layers.Conv1D(filters=512, kernel_size=20, activation='relu',input_shape=(530,2)))
    network_cnn_1.add(layers.Conv1D(filters=256, kernel_size=20, activation='relu'))
    network_cnn_1.add(layers.GlobalMaxPooling1D())
    network_cnn_1.add(layers.Flatten())
    network_cnn_1.add(layers.Dense(1024, activation='relu', input_shape=(6000,)))
    network_cnn_1.add(layers.Dense(512, activation='relu'))
    network_cnn_1.add(layers.Dense(256, activation='relu'))
    network_cnn_1.add(layers.Dense(128, activation='relu'))
    network_cnn_1.add(layers.Dense(64, activation='relu'))
    network_cnn_1.add(layers.Dense(1, activation='sigmoid'))
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.0001)
    network_cnn_1.compile(optimizer=optimizer,
                loss='binary_crossentropy',
                metrics=['accuracy'])


    return network_cnn_1


In [ ]:
def net_cnn_2():
    #tf.keras.backend.clear_session()
    network_cnn_2 = models.Sequential()
    network_cnn_2.add(layers.Conv1D(filters=1024, kernel_size=k_size, activation='relu',input_shape=(530,2)))
    network_cnn_2.add(layers.Conv1D(filters=512, kernel_size=k_size, activation='relu'))
    network_cnn_2.add(layers.Conv1D(filters=256, kernel_size=k_size, activation='relu'))
    #network_cnn_2.add(layers.Conv1D(filters=, kernel_size=k_size, activation='relu'))
    network_cnn_2.add(layers.GlobalMaxPooling1D())
    network_cnn_2.add(layers.Flatten())

    #network.add(layers.Dense(16, activation='relu'))
    network_cnn_2.add(layers.Dense(1024, activation='relu'))
    network_cnn_2.add(layers.Dense(512, activation='relu'))
    network_cnn_2.add(layers.Dense(256, activation='relu'))
    network_cnn_2.add(layers.Dense(128, activation='relu'))
    network_cnn_2.add(layers.Dense(64, activation='relu'))
    network_cnn_2.add(layers.Dense(32, activation='relu'))
    #network.add(tf.keras.layers.Dropout(0.5))
    #network.add(tf.keras.layers.Dense(128, activation='relu',activity_regularizer=tf.keras.regularizers.l1(0.01)))
    network_cnn_2.add(layers.Dense(1, activation='sigmoid'))
    network_cnn_2.summary()
    network_cnn_2.compile(optimizer=Adam(learning_rate=0.0001),
                loss='binary_crossentropy',
                metrics=['accuracy'])
    return network_cnn_2


## Dense NN

In [ ]:
def net_dense_1():


    #Clasificacion
    input1= tf.keras.Input((530,2),name='Spectra')
    l=layers.Flatten(input_shape=(530, 2))(input1)
    l = Dense(1024, activation='relu')(l)
    l = Dense(512, activation='relu')(l)
    l = Dropout(drop)(l)
    l = Dense(256, activation='relu')(l)
    l = Dropout(drop)(l)
    l = Dense(128, activation='relu')(l)
    l = Dense(64, activation='relu')(l)
    l = Dropout(drop)(l)
    l = Dense(64, activation='relu')(l)
    l = Dropout(drop)(l)
    
    l = Dense(32, activation='relu')(l)
    l = Dense(16, activation='relu')(l)
    classif = Dense(1, activation='sigmoid',name='class_out')(l)

    # Redshift

    merged_layers = layers.concatenate([classif,l])

    l2= layers.Dense(8,activation='relu')(merged_layers)
    
    l2= layers.Dense(4,activation='relu')(l2)
    l2= layers.Dense(2,activation='relu')(l2)
    redshift= layers.Dense(1,name='redshift_out')(l2)




    model_denso_1 = tf.keras.Model(
    inputs=input1,
    outputs=[classif,redshift]
    )

  
    optimizer = tf.keras.optimizers.legacy.Adam(0.0001)
    model_denso_1.compile(optimizer=optimizer,
                loss={'redshift_out': 'mean_absolute_error', 
                    'class_out': 'binary_crossentropy'},
                metrics={'class_out': 'accuracy','redshift_out':'mae'})
    
    return model_denso_1



In [ ]:
def net_dense_2():
    #Clasificacion
    input1= tf.keras.Input((530,2),name='Spectra')
    l=layers.Flatten(input_shape=(530, 2))(input1)
    l = Dense(512, activation='relu')(l)
    l = Dense(256, activation='relu')(l)
    l = Dense(128, activation='relu')(l)
    l = Dropout(drop)(l)
    l = Dense(64, activation='relu')(l)
    l = Dropout(drop)(l)
    #model.add(tf.keras.layers.Dropout(0.4))
    l = Dense(32, activation='relu')(l)
    l = Dense(16, activation='relu')(l)
    classif = Dense(1, activation='sigmoid',name='class_out')(l)

    # Redshift
    merged_layers = layers.concatenate([classif,l])
    l2= layers.Dense(8,activation='relu')(merged_layers)
    #l2 = layers.BatchNormalization()(l2)
    l2= layers.Dense(4,activation='relu')(l2)
    l2= layers.Dense(2,activation='relu')(l2)
    redshift= layers.Dense(1,name='redshift_out')(l2)


    model_denso_2 = tf.keras.Model(
        inputs=input1,
        outputs=[classif,redshift]
    )

    optimizer = tf.keras.optimizers.legacy.Adam(0.0001)
    model_denso_2.compile(optimizer=optimizer,
                loss={'redshift_out': 'mean_absolute_error', 
                    'class_out': 'binary_crossentropy'},
                metrics={'class_out': 'accuracy','redshift_out':'mae'})
    
    return model_denso_2



# Keras Tuner models

In [ ]:
def net_keras_1():

    #Clasificacion
    input1= tf.keras.Input((530,2),name='Spectra')
    l=layers.Flatten(input_shape=(530, 2))(input1)
    l = Dense(64, activation='relu')(l)
    #l = layers.BatchNormalization()(l)
    l = Dense(960, activation='relu')(l)
    l = Dense(160, activation='relu')(l)
    l = Dense(960, activation='relu')(l)
    l = Dense(896, activation='relu')(l)
    classif = Dense(1, activation='sigmoid',name='class_out')(l)
    merged_layers = layers.concatenate([classif,l])
    l2= layers.Dense(8,activation='relu')(merged_layers)
    l2= layers.Dense(4,activation='relu')(l2)
    l2= layers.Dense(2,activation='relu')(l2)
    redshift= layers.Dense(1,name='redshift_out')(l2)




    model_denso_4 = tf.keras.Model(
    inputs=input1,
    outputs=[classif,redshift]
    )


    #tf.keras.utils.plot_model(model_denso_4,show_shapes=True,dpi=100,rankdir='TB')


    #tf.keras.backend.clear_session()
    optimizer = tf.keras.optimizers.legacy.Adam(0.0001)
    model_denso_4.compile(optimizer=optimizer,
                loss={'redshift_out': 'mean_absolute_error', 
                    'class_out': 'binary_crossentropy'},
                metrics={'class_out': 'accuracy','redshift_out':'mae'})
    return model_denso_4


In [ ]:
def net_keras_2():
    #Clasificacion
    input1= tf.keras.Input((530,2),name='Spectra')
    l=layers.Flatten(input_shape=(530, 2))(input1)
    l = Dense(768, activation='relu')(l)
   
    l = Dense(1024, activation='relu')(l)
    
    l = Dense(672, activation='relu')(l)

    l = Dense(480, activation='relu')(l)
    l = Dense(544, activation='relu')(l)

    classif = Dense(1, activation='sigmoid',name='class_out')(l)

    # Redshift
    
    merged_layers = layers.concatenate([classif,l])
    l2= layers.Dense(8,activation='relu')(merged_layers)
    #l2 = layers.BatchNormalization()(l2)
    l2= layers.Dense(4,activation='relu')(l2)
    l2= layers.Dense(2,activation='relu')(l2)
    redshift= layers.Dense(1,name='redshift_out')(l2)




    model_denso_3 = tf.keras.Model(
    inputs=input1,
    outputs=[classif,redshift]
    )


    #tf.keras.utils.plot_model(model_denso_3,show_shapes=True,dpi=100,rankdir='TB')


    #tf.keras.backend.clear_session()
    optimizer = tf.keras.optimizers.legacy.Adam(0.0001)
    model_denso_3.compile(optimizer=optimizer,
                loss={'redshift_out': 'mean_absolute_error', 
                    'class_out': 'binary_crossentropy'},
                metrics={'class_out': 'accuracy','redshift_out':'mae'})
    
    return model_denso_3


# Meta learner

In [ ]:
# Meta learnermodel_stacking = Sequential([
    Dense(10, activation='relu', input_shape=(6,)),  # Capa oculta
    Dense(1, activation='sigmoid')  # Salida binaria, usa 'softmax' si hay más clases
])
model_stacking.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Execution

In [ ]:
# =========================
# Configuración general
# =========================
bootstrap_size = 1600  # Tamaño de la muestra bootstrap
bootstrap_test = 389
threshold = 0.5
class_names = ["LT", "ET"]
output_dir = "Resultados_Modelos_v16_std_v2"
outliers_path = os.path.join(output_dir, "Outliers.csv")  # Archivo con 3 columnas

# =========================
# Utilidades
# =========================
def ensure_vector(x):
    """Convierte salidas (listas, tuplas, arrays, (n,1), etc.) a un vector 1D np.float64."""
    x = np.asarray(x)
    if x.ndim > 1:
        x = x.reshape(-1)
    return x.astype(np.float64)

def crear_directorio(base_path, iteracion, sub_iteracion, modelo):
    path = os.path.join(base_path, f"iter_{iteracion}_sub_{sub_iteracion}", modelo)
    os.makedirs(path, exist_ok=True)
    return path

def guardar_roc(y_true, y_pred, save_path, title):
    y_true = ensure_vector(y_true).astype(int)
    y_pred = ensure_vector(y_pred).astype(float)

    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    roc_auc = auc(fpr, tpr)

    # Índice de Youden (maximiza TPR - FPR)
    youden_index = int(np.argmax(tpr - fpr))
    youden_threshold = float(thresholds[youden_index])
    youden_fpr, youden_tpr = float(fpr[youden_index]), float(tpr[youden_index])

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.3f}')
    plt.scatter(youden_fpr, youden_tpr, marker='o', label=f'Youden = {youden_threshold:.3f}')
    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid()
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(os.path.join(save_path, f'{title}_roc.png'), dpi=300)
    plt.close()

    # Guardar CSV con columnas FPR,TPR,Threshold
    roc_data = np.column_stack((fpr, tpr, thresholds))
    np.savetxt(os.path.join(save_path, f'{title}_roc.csv'), roc_data, delimiter=',',
               header='FPR,TPR,Threshold', comments='')

    return roc_auc, youden_threshold

def guardar_confusion_matrix(y_true, y_pred_labels, class_names, save_path, title):
    y_true = ensure_vector(y_true).astype(int)
    y_pred_labels = ensure_vector(y_pred_labels).astype(int)
    cm = confusion_matrix(y_true, y_pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(os.path.join(save_path, f'{title}_confusion_matrix.png'), dpi=300)
    plt.close()

def guardar_roc_comparativa(y_true, dict_preds, save_path, title):
    y_true = ensure_vector(y_true).astype(int)
    plt.figure(figsize=(8, 6))
    for modelo, pred in dict_preds.items():
        pred = ensure_vector(pred).astype(float)
        fpr, tpr, thresholds = roc_curve(y_true, pred)
        roc_auc = auc(fpr, tpr)

        # Índice de Youden
        youden_index = int(np.argmax(tpr - fpr))
        youden_threshold = float(thresholds[youden_index])
        youden_fpr, youden_tpr = float(fpr[youden_index]), float(tpr[youden_index])

        label_name = str(modelo).replace("_", " ")
        plt.plot(fpr, tpr, lw=2, label=f'{label_name} (AUC = {roc_auc:.3f})')
        plt.scatter(youden_fpr, youden_tpr, marker='o', s=18, label=f'{label_name} Youden={youden_threshold:.3f}')

    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(title)
    plt.legend(loc="lower right", fontsize=8)
    plt.grid()
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(os.path.join(save_path, f"{title}_roc_comparativa.png"), dpi=300)
    plt.close()

# ==== Guardado de IDs en tres columnas ====
def append_three_columns(out_csv_path, ids_05, ids_youden, ids_geom):
    """
    Apendea al CSV filas con tres columnas:
    [mis_0.5, mis_Youden, mis_Geom]
    Si las listas tienen longitudes distintas, se rellenan con vacío.
    Crea encabezado si el archivo no existe.
    """
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    file_exists = os.path.isfile(out_csv_path)
    max_len = max(len(ids_05), len(ids_youden), len(ids_geom))
    with open(out_csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(["mis_0.5", "mis_Youden", "mis_Geom"])
        for i in range(max_len):
            c0 = ids_05[i]     if i < len(ids_05)     else ""
            c1 = ids_youden[i] if i < len(ids_youden) else ""
            c2 = ids_geom[i]   if i < len(ids_geom)   else ""
            w.writerow([c0, c1, c2])

# ==== Modelo stacking con ponderación ====
def build_stacking_model():
    prior_weights = tf.constant([0.1676, 0.1642, 0.1668, 0.1661, 0.1670, 0.1683], dtype=tf.float32)
    inp = Input(shape=(6,), name="stack_inputs")
    scaled = Lambda(lambda x: x * prior_weights, name="prior_weighting")(inp)

    x = BatchNormalization()(scaled)
    x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)
    x = Dense(16, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inp, out, name="ImprovedStacking")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ==== Calibración tipo Platt para cada modelo base ====
def calibrar_predicciones_con_platt(modelos_train, modelos_test, y_train_binaria):
    """
    modelos_train: dict nombre -> vector de probas (n_train,)
    modelos_test : dict nombre -> vector de probas (n_test,)
    y_train_binaria: (n_train,) etiquetas 0/1 para meta-train
    Devuelve: (dict_cal_train, dict_cal_test) con probas calibradas por modelo.
    """
    y = ensure_vector(y_train_binaria).astype(int)
    cal_train = {}
    cal_test = {}

    for name in modelos_train.keys():
        p_tr = ensure_vector(modelos_train[name]).astype(float)
        p_te = ensure_vector(modelos_test[name]).astype(float)

        # Platt scaling: regresión logística 1D
        lr = LogisticRegression(max_iter=1000)
        lr.fit(p_tr.reshape(-1, 1), y)
        p_tr_cal = lr.predict_proba(p_tr.reshape(-1, 1))[:, 1]
        p_te_cal = lr.predict_proba(p_te.reshape(-1, 1))[:, 1]

        cal_train[name] = p_tr_cal
        cal_test[name] = p_te_cal

    return cal_train, cal_test

# =========================
# Bucle principal
# =========================
# IMPORTANTE: Se asume que ya existen:
# x_train, y, z_t, x_lambda, spcid_test
# modelo_1(), net_cnn_1(), net_cnn_2(), net_dense_1(), net_dense_2(), net_keras_1(), net_keras_2(), net_gru(), net_lstm()
# reduce_lr, reduce_lr_1, early_stopping, early_stopping_1
# Asegúrate de tenerlos definidos antes de ejecutar este script.

os.makedirs(output_dir, exist_ok=True)

for s in range(6):
    indices = np.random.permutation(len(x_train))
    x_train_mixed = x_train[indices]
    y_train_mixed = y[indices]
    z_train_mixed = z_t[indices]
    lambda_mixed = x_lambda[indices]
    spcid_mixed = spcidr[indices]  # Alinear IDs con la mezcla

    # Bootstrap train
    bootstrap_indices = np.random.choice(len(x_train_mixed[:1600]), size=bootstrap_size, replace=True)
    xx_train, yy_train, zz_train, lambda_train = (
        x_train_mixed[bootstrap_indices],
        y_train_mixed[bootstrap_indices],
        z_train_mixed[bootstrap_indices],
        lambda_mixed[bootstrap_indices],
    )

    # Bootstrap test (del rango 1600:1985)
    bootstrap_indices_test = np.random.choice(np.arange(1600, 1985), size=bootstrap_test, replace=True)
    xx_test, yy_test, zz_test, lambda_test = (
        x_train_mixed[bootstrap_indices_test],
        y_train_mixed[bootstrap_indices_test],
        z_train_mixed[bootstrap_indices_test],
        lambda_mixed[bootstrap_indices_test],
    )
    # IDs del test para esta corrida
    spcid_test_boot = spcid_mixed[bootstrap_indices_test]

    tf.keras.backend.clear_session()

    for k in range(2):
        tf.keras.backend.clear_session()

        # ==== Definición de modelos base (debes tenerlas implementadas) ====
        model = modelo_1()
        network_cnn_1 = net_cnn_1()
        network_cnn_2 = net_cnn_2()
        model_denso_1 = net_dense_1()
        model_denso_2 = net_dense_2()
        model_denso_3 = net_keras_1()
        model_denso_4 = net_keras_2()
        network_gru = net_gru()
        network_lstm = net_lstm()

        # ==== Modelo stacking ====
        model_stacking = build_stacking_model()

        # ==== Preprocesamiento de datos ====
        history = model.fit(
            xx_train,
            {'class_out': yy_train, 'redshift_out': zz_train},
            epochs=100, batch_size=128, callbacks=[reduce_lr], verbose=0
        )

        # Predicciones del modelo multitarea (para correcciones)
        z_nn_train = model.predict(xx_train, verbose=0)
        z_nn_test  = model.predict(xx_test,  verbose=0)

        z_nn_train_red = z_nn_train[1] if isinstance(z_nn_train, (list, tuple)) else z_nn_train
        z_nn_test_red  = z_nn_test[1]  if isinstance(z_nn_test,  (list, tuple)) else z_nn_test

        xx_train_c = [xx_train[r] * (1 + z_nn_train_red[r]) for r in range(len(zz_train))]
        lambda_train_c = [lambda_train[r] / (1 + z_nn_train_red[r]) for r in range(len(zz_train))]

        xx_test_c = [xx_test[p] * (1 + z_nn_test_red[p]) for p in range(len(zz_test))]
        lambda_test_c = [lambda_test[p] / (1 + z_nn_test_red[p]) for p in range(len(zz_test))]

        xx_train_corrected = np.array(xx_train_c)
        xx_test_corrected  = np.array(xx_test_c)
        lambda_train_corrected = np.array(lambda_train_c)
        lambda_test_corrected  = np.array(lambda_test_c)

        # === CORREGIDO: usar axis=-1 correctamente ===
        xx_train_augmented = np.stack([xx_train_corrected, lambda_train_corrected], axis=-1)
        xx_test_augmented  = np.stack([xx_test_corrected,  lambda_test_corrected],  axis=-1)

        # Invierte ejes para que [:,:,0] sea longitud de onda normalizada
        xx_train_augmented_2 = xx_train_augmented[..., ::-1]
        xx_test_augmented_2  = xx_test_augmented[...,  ::-1]

        wavelengths_train = xx_train_augmented_2[:, :, 0]
        wavelengths_test  = xx_test_augmented_2[:,  :, 0]

        wavelengths_global_min = np.min(np.concatenate([wavelengths_train, wavelengths_test]))
        wavelengths_global_max = np.max(np.concatenate([wavelengths_train, wavelengths_test]))
        denom = max(wavelengths_global_max - wavelengths_global_min, 1e-12)

        xx_train_augmented_2[:, :, 0] = (wavelengths_train - wavelengths_global_min) / denom
        xx_test_augmented_2[:,  :, 0] = (wavelengths_test  - wavelengths_global_min) / denom

        # ==== Entrenamiento de cada modelo base ====
        print('Entrenando CNN_1')
        network_cnn_1.fit(
            xx_train_augmented_2[:1400], yy_train[:1400],
            epochs=60, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600], yy_train[1400:1600]),
            callbacks=[reduce_lr_1, early_stopping], verbose=0
        )

        print('Entrenando CNN_2')
        network_cnn_2.fit(
            xx_train_augmented_2[:1400], yy_train[:1400],
            epochs=100, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600], yy_train[1400:1600]),
            callbacks=[reduce_lr_1, early_stopping], verbose=0
        )

        print('Entrenando Dense_1')
        model_denso_1.fit(
            xx_train_augmented_2[:1400],
            {'class_out': yy_train[:1400], 'redshift_out': zz_train[:1400]},
            epochs=100, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600],
                             {'class_out': yy_train[1400:1600], 'redshift_out': zz_train[1400:]}),
            callbacks=[reduce_lr, early_stopping_1], verbose=0
        )

        print('Entrenando Dense_2')
        model_denso_2.fit(
            xx_train_augmented_2[:1400],
            {'class_out': yy_train[:1400], 'redshift_out': zz_train[:1400]},
            epochs=100, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600],
                             {'class_out': yy_train[1400:1600], 'redshift_out': zz_train[1400:]}),
            callbacks=[reduce_lr, early_stopping_1], verbose=0
        )

        print('Entrenando Dense_3')
        model_denso_3.fit(
            xx_train_augmented_2[:1400],
            {'class_out': yy_train[:1400], 'redshift_out': zz_train[:1400]},
            epochs=100, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600],
                             {'class_out': yy_train[1400:1600], 'redshift_out': zz_train[1400:]}),
            callbacks=[reduce_lr, early_stopping_1], verbose=0
        )

        print('Entrenando Dense_4')
        model_denso_4.fit(
            xx_train_augmented_2[:1400],
            {'class_out': yy_train[:1400], 'redshift_out': zz_train[:1400]},
            epochs=100, batch_size=128,
            validation_data=(xx_train_augmented_2[1400:1600],
                             {'class_out': yy_train[1400:1600], 'redshift_out': zz_train[1400:]}),
            callbacks=[reduce_lr, early_stopping_1], verbose=0
        )

        # RNNs
        xx_train_lstm = xx_train_augmented_2[..., 1].reshape(-1, 530, 1)
        print('GRU')
        network_gru.fit(xx_train_lstm[:1400], yy_train[:1400], epochs=30, batch_size=64, verbose=0)

        print('LSTM')
        network_lstm.fit(xx_train_lstm[:1400], yy_train[:1400], epochs=30, batch_size=64, verbose=0)

        xx_test_lstm = xx_test_augmented_2[..., 1].reshape(-1, 530, 1)

        # === Predicciones de modelos base en TEST ===
        modelos = {
            "CNN_1": ensure_vector(network_cnn_1.predict(xx_test_augmented_2, verbose=0)),
            "CNN_2": ensure_vector(network_cnn_2.predict(xx_test_augmented_2, verbose=0)),
            "Dense_1": ensure_vector(model_denso_1.predict(xx_test_augmented_2, verbose=0)[0]),
            "Dense_2": ensure_vector(model_denso_2.predict(xx_test_augmented_2, verbose=0)[0]),
            "Dense_Tuner_1": ensure_vector(model_denso_3.predict(xx_test_augmented_2, verbose=0)[0]),
            "Dense_Tuner_2": ensure_vector(model_denso_4.predict(xx_test_augmented_2, verbose=0)[0]),
        }

        # === Predicciones de modelos base en TRAIN (para stacking) ===
        modelos_train = {
            "CNN_1": ensure_vector(network_cnn_1.predict(xx_train_augmented_2[:1400], verbose=0)),
            "CNN_2": ensure_vector(network_cnn_2.predict(xx_train_augmented_2[:1400], verbose=0)),
            "Dense_1": ensure_vector(model_denso_1.predict(xx_train_augmented_2[:1400], verbose=0)[0]),
            "Dense_2": ensure_vector(model_denso_2.predict(xx_train_augmented_2[:1400], verbose=0)[0]),
            "Dense_Tuner_1": ensure_vector(model_denso_3.predict(xx_train_augmented_2[:1400], verbose=0)[0]),
            "Dense_Tuner_2": ensure_vector(model_denso_4.predict(xx_train_augmented_2[:1400], verbose=0)[0]),
        }

        # === Preparar datos para el stacking ===
        y_meta_train = yy_train[:1400].astype(int)

        # Calibración Platt por modelo
        modelos_train_calibrados, modelos_test_calibrados = calibrar_predicciones_con_platt(
            modelos_train, modelos, y_meta_train
        )

        X_meta_train_calibrado = np.column_stack([modelos_train_calibrados[k] for k in sorted(modelos_train_calibrados.keys())])
        X_meta_test_calibrado  = np.column_stack([modelos_test_calibrados[k]  for k in sorted(modelos_test_calibrados.keys())])

        # Entrenamiento del stacking
        model_stacking.fit(X_meta_train_calibrado, y_meta_train, epochs=30, batch_size=32, verbose=0)

        # Predicción en test
        pred_stacking = ensure_vector(model_stacking.predict(X_meta_test_calibrado, verbose=0))

        # === Guardado de métricas y curvas ===
        for modelo_name, pred in {**modelos, "Model_stacking": pred_stacking}.items():
            path_modelo = crear_directorio(output_dir, s, k, modelo_name)

            # Bloque 1
            y_true_b1 = yy_test[:194]
            y_pred_b1 = pred[:194]
            auc_block1, youden_b1 = guardar_roc(y_true_b1, y_pred_b1, path_modelo, "block1")
            guardar_confusion_matrix(y_true_b1, (y_pred_b1 > threshold).astype(int), class_names, path_modelo, "block1")
            acc_block1 = accuracy_score(y_true_b1.astype(int), (y_pred_b1 > threshold).astype(int))

            # Bloque 2
            y_true_b2 = yy_test[194:]
            y_pred_b2 = pred[194:]
            auc_block2, youden_b2 = guardar_roc(y_true_b2, y_pred_b2, path_modelo, "block2")
            guardar_confusion_matrix(y_true_b2, (y_pred_b2 > threshold).astype(int), class_names, path_modelo, "block2")
            acc_block2 = accuracy_score(y_true_b2.astype(int), (y_pred_b2 > threshold).astype(int))

            # Guardar predicciones crudas
            with open(os.path.join(path_modelo, "pred_block1.csv"), 'w', encoding='UTF8', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["True", "Pred"])
                writer.writerows(zip(ensure_vector(y_true_b1).astype(int), ensure_vector(y_pred_b1).astype(float)))

            with open(os.path.join(path_modelo, "pred_block2.csv"), 'w', encoding='UTF8', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["True", "Pred"])
                writer.writerows(zip(ensure_vector(y_true_b2).astype(int), ensure_vector(y_pred_b2).astype(float)))

            # Guardar métricas
            with open(os.path.join(path_modelo, "metrics.csv"), 'w', encoding='UTF8', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Epocas", "Accuracy_B1", "Accuracy_B2", "AUC_B1", "AUC_B2", "Youden_B1", "Youden_B2"])
                writer.writerow([30, acc_block1, acc_block2, auc_block1, auc_block2, youden_b1, youden_b2])

        # === Curva ROC comparativa
        dict_preds = {**modelos, "Model_stacking": pred_stacking}
        guardar_roc_comparativa(yy_test, dict_preds, output_dir, f"ROC_Comparativa_iter{s}_sub{k}")

        # ============================================================
        # Guardar IDs mal clasificados del STACKING en 3 columnas
        #        (0.5, Youden, Geométrico), apendando por corrida
        # ============================================================
        y_true_full = yy_test.astype(int)

        # 1) 0.5 fijo
        labels_05 = (pred_stacking > 0.5).astype(int)
        mis_idx_05 = np.where(labels_05 != y_true_full)[0]
        ids_05 = list(spcid_test_boot[mis_idx_05])

        # 2) Youden
        fpr, tpr, thr = roc_curve(y_true_full, pred_stacking)
        youden_idx = int(np.argmax(tpr - fpr))
        thr_youden = float(thr[youden_idx])
        labels_y = (pred_stacking >= thr_youden).astype(int)
        mis_idx_y = np.where(labels_y != y_true_full)[0]
        ids_youden = list(spcid_test_boot[mis_idx_y])

        # 3) Geométrico (mín. distancia a (0,1))
        d2 = (fpr - 0.0)**2 + (1.0 - tpr)**2
        geom_idx = int(np.argmin(d2))
        thr_geom = float(thr[geom_idx])
        labels_g = (pred_stacking >= thr_geom).astype(int)
        mis_idx_g = np.where(labels_g != y_true_full)[0]
        ids_geom = list(spcid_test_boot[mis_idx_g])

        # Append al CSV (3 columnas)
        append_three_columns(outliers_path, ids_05, ids_youden, ids_geom)

      
